# Module 4: Designing a Vector Search System

You know the parts from Modules 1 to 3: embeddings, collections, HNSW, hybrid retrieval, and filters. This module is about judgment, going from "we have articles and analysts" to a payload schema, a retrieval pipeline, and a deployment mode.

## What You Will Do

1. See the five layers of the stack as a diagnostic checklist.
2. Design and build a news search system by answering five questions.
3. Filter the way production systems do, and see what happens when you get it wrong.
4. Assemble a production RAG retrieval pipeline.
5. Review the deployment options and work through a knowledge check.

**Running this in Colab:** run the cells top to bottom. The first two cells take a minute, because they download the embedding models.

## Setup

The notebook runs against Qdrant in **local mode**, an in-memory Qdrant that needs no server. It is the fastest way to follow along, and it has one limit worth knowing before you start: local mode is a Python reimplementation, not the engine. Payload indexes have no effect there, and search is exact rather than approximate. Every index call below is still the right habit, and Section 5 covers when to move off it.

To run against a real cluster instead, create a free one at [cloud.qdrant.io](https://cloud.qdrant.io/), then swap in the commented lines below.

In [ ]:
!pip install -q "qdrant-client[fastembed]"

In [2]:
from qdrant_client import QdrantClient, models
from fastembed import TextEmbedding, SparseTextEmbedding

# Local mode: in-memory Qdrant, no server needed.
client = QdrantClient(":memory:")

# To use a real cluster instead, comment out the line above and uncomment these.
# Colab stores secrets under the key icon in the left sidebar.
#
# from google.colab import userdata
# client = QdrantClient(
#     url=userdata.get("QDRANT_URL"),
#     api_key=userdata.get("QDRANT_API_KEY"),
# )

# A small, fast English model. Any embedding model works, this one keeps the
# Colab download short. It is also FastEmbed's default.
dense_model = TextEmbedding("BAAI/bge-small-en-v1.5")

# BM25 gives us exact-token matching alongside dense semantics.
sparse_model = SparseTextEmbedding("Qdrant/bm25")

# Ask the model for its dimensionality rather than hardcoding it, so the
# collection config can never drift out of sync with the model.
DENSE_DIM = len(list(dense_model.embed(["hello"]))[0])
print("Dense vector dimension:", DENSE_DIM)

Dense vector dimension: 384


## 1. The Layers of the Stack

Every vector search system is built from the same five layers. When something is slow, wrong, or expensive, the first question is always which layer the problem is in.

| Layer | What lives here |
|---|---|
| Query | Embedding the query, dense vs. sparse vs. hybrid, fusion, limits |
| Indexing | The HNSW graph for vectors, payload indexes for filter fields |
| Storage | Vectors, payloads, and IDs on disk and in memory |
| Knowledge | The data itself: chunking, embedding model choice, payload schema |
| Distribution | Sharding, replication, multi-node clusters |

Every decision below is tagged with the layer it belongs to.

## 2. Worked Example: Designing a News Search System

> Analysts at a research firm need to search global news that arrives continuously. They ask in plain language ("port congestion in Southeast Asia") and they scope every search by country, topic, date range, and source. Some queries name one specific thing, a company ticker or a ship name, that has to match exactly.

Five questions turn that brief into a system.

### Question 1: What Do the Queries Look Like?

Both kinds. "Port congestion in Southeast Asia" is semantic intent, which is dense territory. "MAERSK-B.CO" is an exact token that carries no meaning a dense model can use.

**Decision:** hybrid search, with a named dense vector and a named sparse vector on every point, fused at query time. *(Query layer.)*

Naming the vectors is what lets a single point carry both and a single query use both. The `modifier=IDF` setting is required for correct BM25 scoring: sparse vectors store term frequency, and Qdrant applies the inverse document frequency half at query time.

In [3]:
client.create_collection(
    collection_name="news",
    vectors_config={
        "dense": models.VectorParams(size=DENSE_DIM, distance=models.Distance.COSINE),
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(
            # Required for correct BM25 scoring
            modifier=models.Modifier.IDF
        ),
    },
)
print("Collection 'news' created.")

Collection 'news' created.


### Question 2: What Must the System Filter On?

From the brief: country, topic, date range, and source. These are hard rules, not ranking signals. An analyst scoping to "Vietnam, last seven days" means exactly that. Hard rules go in the payload, and every field you filter on gets a payload index.

The schema, decided now, before ingestion:

```
payload:
  country       string    (indexed)
  topic         string    (indexed)
  source        string    (indexed)
  published_at  datetime  (indexed)
  headline      string    (returned, never filtered)
  lead          string    (returned, never filtered)
  body          string    (returned, never filtered)
```

That is a knowledge-layer decision (what to store) and an indexing-layer decision (what to index).

The next cell prints a warning: `Payload indexes have no effect in the local Qdrant`. That is expected and it is a local-mode artifact, not a mistake. Creating the indexes before ingestion is the correct habit, and on a real cluster it is what makes filtered search fast.

In [4]:
for field in ["country", "topic", "source"]:
    client.create_payload_index(
        collection_name="news",
        field_name=field,
        field_schema=models.PayloadSchemaType.KEYWORD,
    )

client.create_payload_index(
    collection_name="news",
    field_name="published_at",
    field_schema=models.PayloadSchemaType.DATETIME,
)
print("Payload indexes created for country, topic, source, published_at.")

Payload indexes created for country, topic, source, published_at.


/tmp/ipykernel_576/4200797916.py:2: UserWarning: Payload indexes have no effect in the local Qdrant. Please use server Qdrant if you need payload indexes.
  client.create_payload_index(


### Question 3: What Is the Workload Shape?

Millions of articles, text only, arriving continuously, and analysts expect this morning's news to be searchable this morning.

**Decisions:** one collection, and continuous upserts rather than periodic rebuilds. *(Storage and knowledge layers.)*

Here is a small sample standing in for that stream. Note the exact tokens hiding in the text: a ticker and a ship name.

In [5]:
articles = [
    {
        "headline": "Hai Phong container backlog grows for a third week",
        "lead": "Congestion at the northern Vietnamese port deepened as container volumes climbed, delaying vessel departures.",
        "body": "Terminal operators said yard utilization reached the high nineties, forcing arriving vessels to wait at anchor. Freight forwarders reported delays of two to four days on outbound bookings.",
        "country": "VN", "topic": "shipping", "source": "reuters",
        "published_at": "2026-07-20T08:00:00Z", "tenant_id": "asia-desk",
    },
    {
        "headline": "Southeast Asian port congestion slows regional transport",
        "lead": "Delays at major Southeast Asian ports are pushing back container ship arrivals across the region.",
        "body": "Analysts point to a combination of higher volumes and reduced berth availability. Shipping lines have added waiting time to their schedules.",
        "country": "JP", "topic": "shipping", "source": "nikkei",
        "published_at": "2026-07-21T09:30:00Z", "tenant_id": "asia-desk",
    },
    {
        "headline": "Shanghai port sets a monthly logistics throughput record",
        "lead": "The Port of Shanghai handled record volumes this month as operators expanded yard capacity.",
        "body": "Terminal managers credited longer gate hours and a new rail link for the improvement. Throughput growth has outpaced the national average for six consecutive quarters.",
        "country": "CN", "topic": "logistics", "source": "caixin",
        "published_at": "2026-07-19T11:00:00Z", "tenant_id": "asia-desk",
    },
    {
        "headline": "Shipping group unit faces delisting speculation",
        "lead": "Shares tied to MAERSK-B.CO swung on speculation that a subsidiary vehicle could be delisted.",
        "body": "Traders cited an unusual volume spike in the final hour of trading. The company declined to comment.",
        "country": "DK", "topic": "markets", "source": "reuters",
        "published_at": "2026-07-22T07:15:00Z", "tenant_id": "europe-desk",
    },
    {
        "headline": "Carrier unit faces delisting speculation",
        "lead": "Shares tied to HLAG-D.DE swung on speculation that a subsidiary vehicle could be delisted.",
        "body": "Traders cited unusual volume late in the session. The company declined to comment.",
        "country": "DE", "topic": "markets", "source": "handelsblatt",
        "published_at": "2026-07-22T08:05:00Z", "tenant_id": "europe-desk",
    },
    {
        "headline": "Ever Given rerouted through Singapore to avoid delays",
        "lead": "The container ship Ever Given was rerouted through the Port of Singapore this week.",
        "body": "The diversion adds roughly two days to the voyage but avoids a longer wait at anchor. Singapore has absorbed a growing share of regional transhipment traffic.",
        "country": "SG", "topic": "shipping", "source": "straits-times",
        "published_at": "2026-07-22T10:45:00Z", "tenant_id": "asia-desk",
    },
    {
        "headline": "Logistics operator reports steady quarterly earnings",
        "lead": "A North American logistics operator announced quarterly earnings, citing steady freight demand.",
        "body": "Revenue was broadly flat against the same quarter last year. The company reiterated its full year guidance.",
        "country": "US", "topic": "markets", "source": "press-release-wire",
        "published_at": "2026-06-30T12:00:00Z", "tenant_id": "americas-desk",
    },
    {
        "headline": "Laem Chabang congestion rises on higher container volumes",
        "lead": "Thailand's largest port is handling more containers than its yards were sized for, lengthening waits.",
        "body": "Operators have extended gate hours and added weekend shifts. Exporters report longer lead times on shipments to Europe.",
        "country": "TH", "topic": "shipping", "source": "bangkok-post",
        "published_at": "2026-07-18T06:20:00Z", "tenant_id": "asia-desk",
    },
    {
        "headline": "Hamburg reports handling delays as arrivals climb",
        "lead": "The Port of Hamburg is reporting delays in cargo handling as the number of arriving ships rises.",
        "body": "Dock labor shortages have compounded the problem. Rail connections inland are running close to capacity.",
        "country": "DE", "topic": "logistics", "source": "handelsblatt",
        "published_at": "2026-07-15T14:10:00Z", "tenant_id": "europe-desk",
    },
]
print(len(articles), "sample articles across", len(set(a["country"] for a in articles)), "countries.")

9 sample articles across 8 countries.


One knowledge-layer decision hides in the ingestion step: **what you embed matters as much as how you search it.**

A news article runs 800 words, and the ticker from Question 1 is one token inside it. Embed the whole body and that token is averaged into a vector about shipping in general. Embed the headline and the lead, keep the full text in the payload, and the dense vector stays about one story while the sparse vector still sees every token.

Two of these articles are deliberately hard: they have near-identical headlines and leads, and differ only in the ticker. Watch what that does to the dense retriever later.

In [6]:
def to_sparse_vector(text, is_query=False):
    """Convert text to a Qdrant SparseVector using BM25.
    BM25 scores queries and documents differently, so queries use query_embed."""
    emb = next(sparse_model.query_embed(text)) if is_query else next(sparse_model.embed([text]))
    return models.SparseVector(indices=emb.indices.tolist(), values=emb.values.tolist())

points = []
for i, art in enumerate(articles):
    # The knowledge-layer decision, made concrete: embed headline and lead only.
    embed_text = art["headline"] + ". " + art["lead"]
    points.append(
        models.PointStruct(
            id=i,
            vector={
                "dense": next(dense_model.embed([embed_text])).tolist(),
                "sparse": to_sparse_vector(embed_text),
            },
            payload=art,
        )
    )

client.upsert(collection_name="news", points=points)
print("Upserted", len(points), "points. Collection count:", client.count("news").count)

Upserted

 9 points. Collection count: 9


### Question 4: What Does the Retrieval Pipeline Look Like?

Start with the simplest pipeline that fits the query analysis: hybrid, from Question 1, plus filters, from Question 2, fused with Reciprocal Rank Fusion. No reranker yet.

One `query_points` call does all of it:

- Two `Prefetch` branches, one per named vector, each pulling 50 candidates.
- `RrfQuery`, which merges the two candidate lists by rank rather than by score, so it does not matter that cosine similarity and BM25 live on different scales.
- The filter **inside each `Prefetch`**, so both retrievers search only the valid subset. Where that filter goes matters more than it looks, and the next section shows what happens when it goes somewhere else.

In [7]:
def search(query_text, query_filter=None, limit=5):
    """Hybrid search: dense + sparse prefetch, fused with RRF.
    The filter goes inside each Prefetch, not on the outer query."""
    dense_q = next(dense_model.query_embed(query_text)).tolist()
    sparse_q = to_sparse_vector(query_text, is_query=True)
    response = client.query_points(
        collection_name="news",
        prefetch=[
            models.Prefetch(query=dense_q, using="dense", filter=query_filter, limit=50),
            models.Prefetch(query=sparse_q, using="sparse", filter=query_filter, limit=50),
        ],
        query=models.RrfQuery(rrf=models.Rrf()),
        limit=limit,
    )
    return response.points


def show(results):
    for r in results:
        p = r.payload
        print(f"[{r.score:.4f}] {p['country']} {p['topic']:<9} {p['source']:<18} | {p['headline']}")


print("Query: 'port congestion in Southeast Asia'\n")
show(search("port congestion in Southeast Asia"))

Query: 'port congestion in Southeast Asia'



[1.0000] JP shipping  nikkei             | Southeast Asian port congestion slows regional transport
[0.5833] TH shipping  bangkok-post       | Laem Chabang congestion rises on higher container volumes
[0.5000] SG shipping  straits-times      | Ever Given rerouted through Singapore to avoid delays
[0.4500] VN shipping  reuters            | Hai Phong container backlog grows for a third week
[0.3667] CN logistics caixin             | Shanghai port sets a monthly logistics throughput record


### Try It: Why Hybrid, Not Dense Alone

Now the exact-token case, and the reason Question 1 chose hybrid.

Run the same query three ways: dense only, sparse only, and hybrid. Two articles in the collection have near-identical headlines and leads, differing only in the ticker. Look at the **gaps between the scores**, not just the order.

In [8]:
QUERY = "MAERSK-B.CO delisting"
dense_q = next(dense_model.query_embed(QUERY)).tolist()
sparse_q = to_sparse_vector(QUERY, is_query=True)

print("DENSE ONLY")
for p in client.query_points("news", query=dense_q, using="dense", limit=3).points:
    print(f"  [{p.score:.4f}] {p.payload['lead'][:62]}")

print("\nSPARSE ONLY (BM25)")
for p in client.query_points("news", query=sparse_q, using="sparse", limit=3).points:
    print(f"  [{p.score:.4f}] {p.payload['lead'][:62]}")

print("\nHYBRID (RRF over both)")
show(search(QUERY, limit=3))

DENSE ONLY
  [0.8120] Shares tied to MAERSK-B.CO swung on speculation that a subsidi
  [0.6956] Shares tied to HLAG-D.DE swung on speculation that a subsidiar
  [0.6071] The container ship Ever Given was rerouted through the Port of

SPARSE ONLY (BM25)
  [11.7931] Shares tied to MAERSK-B.CO swung on speculation that a subsidi
  [2.5926] Shares tied to HLAG-D.DE swung on speculation that a subsidiar

HYBRID (RRF over both)
[1.0000] DK markets   reuters            | Shipping group unit faces delisting speculation
[0.6667] DE markets   handelsblatt       | Carrier unit faces delisting speculation
[0.2500] SG shipping  straits-times      | Ever Given rerouted through Singapore to avoid delays


Dense puts the right article first, but only just. The correct article scores 0.8120 and the near-identical decoy scores 0.6956, a gap of 0.12, because a ticker is a string with no semantics for a dense model to work with. BM25 scores them 11.79 and 2.59, separating them by more than 4x, because it is matching the literal token.

On nine articles that thin dense margin still lands the right answer. On nine million, with thousands of similar market stories, it is noise. That gap is the argument for hybrid, and it is why Question 1 is the most consequential question in the list.

### Question 5: What Are the Deployment Constraints?

A research firm with a small engineering team, no data residency restrictions, and a "please do not page us at night" budget points to a managed deployment. The design and the deployment mode are independent decisions: everything above runs unchanged against this in-memory client, a Docker container, or Qdrant Cloud. Section 5 lays out the full option space. *(Distribution layer.)*

### The Design on One Page

| Question | Answer for this system | Layer |
|---|---|---|
| Query type | Mixed semantic and exact, so hybrid with Reciprocal Rank Fusion | Query |
| Filter scope | country, topic, source, date, indexed before ingestion | Knowledge, indexing |
| Workload shape | Millions of text chunks, continuous ingestion | Storage, knowledge |
| Pipeline | Hybrid with per-prefetch filters, headline and lead embedded, no reranker yet | Query, knowledge |
| Deployment | Managed, and independent of the design | Distribution |

## 3. Filtering

Filtering is the feature that decides whether your results are correct, so it deserves more than a passing mention.

The naive approach is post-filtering: retrieve the top K by similarity, then throw away whatever fails the filter. With a selective filter, one country out of 200, the top K can contain zero valid results, and there is no K that guarantees correctness. Qdrant does not work that way. A query planner picks a strategy per segment based on how many points it estimates the filter will match, and on which payload indexes exist.

### The Filter Toolbox

| Condition | Logic | Example |
|---|---|---|
| must | AND, all conditions true | country = VN AND topic = shipping |
| should | OR, at least one true | topic = shipping OR topic = logistics |
| must_not | Exclude matches | Exclude source = press-release-wire |
| Range | Numeric or datetime bounds | published_at within the last seven days |
| MatchAny | Value in a set | source in [reuters, nikkei, caixin] |
| Geo | Radius, bounding box, or polygon | Events within 100 km of a port |

The cells below run several of these against the news collection.

In [9]:
# must: AND. Only Vietnamese shipping news.
f_must = models.Filter(
    must=[
        models.FieldCondition(key="country", match=models.MatchValue(value="VN")),
        models.FieldCondition(key="topic", match=models.MatchValue(value="shipping")),
    ]
)
print("must (country=VN AND topic=shipping):")
show(search("port congestion", query_filter=f_must))

must (country=VN AND topic=shipping):
[1.0000] VN shipping  reuters            | Hai Phong container backlog grows for a third week


In [10]:
# should: OR. Either shipping or logistics.
f_should = models.Filter(
    should=[
        models.FieldCondition(key="topic", match=models.MatchValue(value="shipping")),
        models.FieldCondition(key="topic", match=models.MatchValue(value="logistics")),
    ]
)
print("should (topic=shipping OR topic=logistics):")
show(search("congestion at ports", query_filter=f_should))

should (topic=shipping OR topic=logistics):
[1.0000] JP shipping  nikkei             | Southeast Asian port congestion slows regional transport
[0.5833] TH shipping  bangkok-post       | Laem Chabang congestion rises on higher container volumes
[0.4762] DE logistics handelsblatt       | Hamburg reports handling delays as arrivals climb
[0.4167] VN shipping  reuters            | Hai Phong container backlog grows for a third week
[0.4000] CN logistics caixin             | Shanghai port sets a monthly logistics throughput record


In [11]:
# must_not + Range + MatchAny, all evaluated together during the search.
f_combined = models.Filter(
    must=[
        models.FieldCondition(
            key="published_at",
            range=models.DatetimeRange(gte="2026-07-19T00:00:00Z"),
        ),
        models.FieldCondition(
            key="source",
            match=models.MatchAny(any=["reuters", "nikkei", "caixin", "straits-times"]),
        ),
    ],
    must_not=[
        models.FieldCondition(key="source", match=models.MatchValue(value="press-release-wire")),
    ],
)
print("published since July 19, from a wire we trust, no press releases:")
show(search("port congestion in Southeast Asia", query_filter=f_combined))

published since July 19, from a wire we trust, no press releases:
[1.0000] JP shipping  nikkei             | Southeast Asian port congestion slows regional transport
[0.5833] VN shipping  reuters            | Hai Phong container backlog grows for a third week
[0.5333] SG shipping  straits-times      | Ever Given rerouted through Singapore to avoid delays
[0.4500] CN logistics caixin             | Shanghai port sets a monthly logistics throughput record
[0.1667] DK markets   reuters            | Shipping group unit faces delisting speculation


### Common Mistake: Filters in the Wrong Place

Every search above passed its filter **inside each `Prefetch`**. There is a second place the client will happily accept one: `query_filter`, on the outer query, next to `prefetch`.

Put the filter inside every `Prefetch`. It narrows what each retriever searches, so both come back with 50 candidates that already satisfy the constraint, and it behaves the same way on every deployment mode.

Local mode ignores an outer filter and raises no error, so a notebook can return results that violate its own filter while looking perfectly healthy. The cell below runs the same VN-and-shipping filter both ways so you can see it.

In [12]:
dense_q = next(dense_model.query_embed("port congestion")).tolist()
sparse_q = to_sparse_vector("port congestion", is_query=True)

# The wrong place: query_filter on the outer query, alongside prefetch.
wrong = client.query_points(
    collection_name="news",
    prefetch=[
        models.Prefetch(query=dense_q, using="dense", limit=50),
        models.Prefetch(query=sparse_q, using="sparse", limit=50),
    ],
    query=models.RrfQuery(rrf=models.Rrf()),
    query_filter=f_must,
    limit=5,
).points

print("FILTER ON THE OUTER QUERY, in local mode:")
show(wrong)
violations = [p for p in wrong if p.payload["country"] != "VN" or p.payload["topic"] != "shipping"]
print(f"  -> {len(wrong)} results, {len(violations)} of them break the filter\n")

print("FILTER INSIDE EACH PREFETCH:")
right = search("port congestion", query_filter=f_must)
show(right)
violations = [p for p in right if p.payload["country"] != "VN" or p.payload["topic"] != "shipping"]
print(f"  -> {len(right)} results, {len(violations)} of them break the filter")

FILTER ON THE OUTER QUERY, in local mode:
[1.0000] JP shipping  nikkei             | Southeast Asian port congestion slows regional transport
[0.5833] TH shipping  bangkok-post       | Laem Chabang congestion rises on higher container volumes
[0.4762] DE logistics handelsblatt       | Hamburg reports handling delays as arrivals climb
[0.4167] VN shipping  reuters            | Hai Phong container backlog grows for a third week
[0.4000] CN logistics caixin             | Shanghai port sets a monthly logistics throughput record
  -> 5 results, 4 of them break the filter

FILTER INSIDE EACH PREFETCH:
[1.0000] VN shipping  reuters            | Hai Phong container backlog grows for a third week
  -> 1 results, 0 of them break the filter


### A Special Case: Scoping by User or Tenant

In almost any multi-user product you have to scope every query to one customer's data. The instinct is a collection per customer, which becomes millions of collections and is unmanageable. The pattern instead:

1. Add a `tenant_id` payload field to every point at ingestion. Already done above.
2. Create a payload index on it with `is_tenant=True`, which tells Qdrant this field identifies tenants so it can keep each tenant's data together on disk. Supported for the `keyword` and `uuid` index types.
3. Filter on it at every query. Never omit it.

Steps 1 and 3 alone are already correct. Step 2 is what keeps them fast as the tenant count grows.

In [13]:
client.create_payload_index(
    collection_name="news",
    field_name="tenant_id",
    field_schema=models.KeywordIndexParams(
        type=models.KeywordIndexType.KEYWORD,
        is_tenant=True,
    ),
)


def tenant_search(query_text, tenant_id, limit=5):
    return search(
        query_text,
        query_filter=models.Filter(
            must=[models.FieldCondition(key="tenant_id", match=models.MatchValue(value=tenant_id))]
        ),
        limit=limit,
    )


print("europe-desk view of 'port delays':")
show(tenant_search("port delays", tenant_id="europe-desk"))
print("\nasia-desk view of 'port delays':")
show(tenant_search("port delays", tenant_id="asia-desk"))

europe-desk view of 'port delays':
[1.0000] DE logistics handelsblatt       | Hamburg reports handling delays as arrivals climb
[0.3333] DK markets   reuters            | Shipping group unit faces delisting speculation
[0.2500] DE markets   handelsblatt       | Carrier unit faces delisting speculation

asia-desk view of 'port delays':
[1.0000] JP shipping  nikkei             | Southeast Asian port congestion slows regional transport
[0.5833] VN shipping  reuters            | Hai Phong container backlog grows for a third week
[0.5833] SG shipping  straits-times      | Ever Given rerouted through Singapore to avoid delays
[0.3667] TH shipping  bangkok-post       | Laem Chabang congestion rises on higher container volumes
[0.3667] CN logistics caixin             | Shanghai port sets a monthly logistics throughput record


Two desks, one collection, and neither sees the other's articles.

### Key Insight

Design the payload schema before you ingest, driven by one question: what will I need to filter on? Time, geography, identity, permissions, and status flags are the usual suspects. Adding a payload field later is easy. Discovering at query time that you never stored the publication date means re-ingesting everything.

## 4. The Production RAG Pipeline

Retrieval-Augmented Generation (RAG) retrieves relevant passages from a vector search engine and hands them to a large language model as context, so the model answers from your data instead of only from what it memorized during training.

The production shape, using everything above:

1. **Query understanding:** pull hard constraints (dates, country, topic) into a filter. Embed the query as a dense vector and a sparse vector.
2. **Hybrid retrieval:** dense and sparse prefetch with the filter on each, fused with Reciprocal Rank Fusion. One `query_points` call.
3. **Optional reranking:** a cross-encoder, a model that scores a query and a passage together rather than separately, reorders the top candidates. Add it only when evaluation shows fused results need refinement.
4. **Generation:** the top passages go in as context and the model writes the answer.

### Rule of Thumb

When RAG quality disappoints, improve step 2 before reaching for a bigger model in step 4. Retrieval quality caps answer quality: the model cannot cite a passage it never received.

The cell below runs steps 1, 2, and the prompt assembly for step 4. It needs no API key.

In [14]:
def rag_context(question, query_filter=None, k=3):
    """Steps 1 and 2: retrieve the top-k passages for a question."""
    hits = search(question, query_filter=query_filter, limit=k)
    passages = [
        f"- ({h.payload['source']}, {h.payload['country']}, {h.payload['published_at'][:10]}) "
        f"{h.payload['headline']}. {h.payload['lead']}"
        for h in hits
    ]
    return "\n".join(passages), hits


def build_prompt(question, query_filter=None, k=3):
    context, hits = rag_context(question, query_filter=query_filter, k=k)
    prompt = (
        "Answer the question using only the sources below. Cite the source name.\n\n"
        f"Sources:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    return prompt, hits


prompt, hits = build_prompt("What is happening with port congestion in Southeast Asia?")
print(prompt)

Answer the question using only the sources below. Cite the source name.

Sources:
- (nikkei, JP, 2026-07-21) Southeast Asian port congestion slows regional transport. Delays at major Southeast Asian ports are pushing back container ship arrivals across the region.
- (reuters, VN, 2026-07-20) Hai Phong container backlog grows for a third week. Congestion at the northern Vietnamese port deepened as container volumes climbed, delaying vessel departures.
- (bangkok-post, TH, 2026-07-18) Laem Chabang congestion rises on higher container volumes. Thailand's largest port is handling more containers than its yards were sized for, lengthening waits.

Question: What is happening with port congestion in Southeast Asia?
Answer:


Step 4 is the only part that needs an API key, so it is optional. To run it, add your key to the Colab secrets panel (the key icon in the left sidebar) under the name `LLM_API_KEY`, then uncomment the block.

In [15]:
# Optional step 4: hand the prompt to a model. This cell runs without a key.
#
# from google.colab import userdata
# from anthropic import Anthropic
#
# llm = Anthropic(api_key=userdata.get("LLM_API_KEY"))
# message = llm.messages.create(
#     model="claude-sonnet-4-5",
#     max_tokens=300,
#     messages=[{"role": "user", "content": prompt}],
# )
# print(message.content[0].text)

print("Prompt is ready. Uncomment the block above and add a key to generate an answer.")

Prompt is ready. Uncomment the block above and add a key to generate an answer.


## 5. Deployment Options

The design above runs unchanged on any of these. Which one is right depends on your constraints, not on sophistication.

| Deployment Mode | Use When | Avoid When |
|---|---|---|
| Local Mode | Prototyping, notebooks, CI tests, teaching | Production or benchmarking |
| Docker, self-hosted | Full infrastructure control, air-gapped or regulated environments | You do not yet have monitoring and backups |
| Managed Cloud | Small ops team, standard requirements | Data cannot leave your infrastructure |
| Hybrid Cloud | Data residency or security policy requires your own infrastructure | Managed cloud would do, with less operational overhead |
| Private Cloud or on-prem | Strictest requirements: defense, healthcare, finance | A lighter mode meets your needs |
| Edge | On-device search, offline, very low latency | You need distributed search |

One caveat about local mode, since it is what this notebook used. It is a Python reimplementation rather than the engine: search is exact rather than approximate, payload indexes have no effect, and a filter on the outer query is ignored. Everything in Section 3 is still worth understanding while working locally, but verify indexing and filtering against a real server before trusting numbers from a notebook.

## 6. Knowledge Check

Work through these before starting the capstone in Module 5.

**1. Name the five layers, and place "chunk articles instead of embedding them whole" in the right one.**
Query, indexing, storage, knowledge, distribution. Chunking is a knowledge-layer decision, and mistakes there cannot be fixed by tuning any other layer.

**2. Why hybrid retrieval from the start, rather than dense only?**
Query analysis showed a mix of semantic intent and exact tokens. A ticker carries no semantics for a dense model, so the dense margin between the right article and a near-identical one is thin. The sparse half separates them decisively.

**3. What is wrong with post-filtering, and how does Qdrant avoid it?**
Post-filtering retrieves the top K first and discards invalid results, so a selective filter can leave zero valid points. Qdrant's query planner instead picks a strategy per segment from the estimated filter cardinality.

**4. In a hybrid query, where does the filter belong?**
Inside each `Prefetch`. It narrows what each retriever searches, and it behaves the same way on every deployment mode. Local mode ignores an outer `query_filter` without raising an error, which is how a notebook ends up printing results that break its own filter.

**5. Why create payload indexes before ingestion rather than after?**
The filterable HNSW graph gains filter-aware edges only for indexes that exist when the graph is built. An index created later still filters and still feeds the planner's cardinality estimate, but getting those extra edges means rebuilding the HNSW index.

**6. Why does per-tenant scoping use a payload filter instead of one collection per tenant?**
Collections do not scale to millions of tenants operationally. An indexed `tenant_id` field, filtered on at every query, gives isolation in one collection. `is_tenant=True` is what keeps it fast as the tenant count grows.

**7. In the RAG pipeline, which step do you improve first when answers disappoint?**
Step 2, retrieval. The model cannot cite a passage it never received.

## What Is Next: Module 5

The capstone extends this system. Same five questions, bigger answers:

- Ingest daily news, audio, and satellite imagery about suppliers, so three modalities instead of one
- Embed each modality into named vectors on shared points
- Cluster signals into risk themes across suppliers

Before you start it, create a free cluster at [cloud.qdrant.io](https://cloud.qdrant.io/) so the capstone runs against a real server rather than local mode.